# People Analytics Turnover - Analysis Notebook

## Objetivo
Identificar risco de saida para apoiar iniciativas de retencao de talentos.

## Perguntas de negocio
- Quais grupos apresentam maior risco?
- Quais fatores sao acionaveis por RH e lideranca?
- Como priorizar acoes sem enviesar decisoes?


### O que este código faz
Carrega bibliotecas, lê os arquivos principais do projeto (`data` e `metrics`) e mostra uma primeira amostra dos dados para conferência inicial.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path('..')
data_path = project_root / 'data' / 'employees_synthetic.csv'
metrics_path = project_root / 'models' / 'metrics.json'

pd.set_option('display.max_columns', 200)
df = pd.read_csv(data_path)
metrics = json.loads(metrics_path.read_text(encoding='utf-8')) if metrics_path.exists() else {}

print('shape:', df.shape)
print('metrics keys:', list(metrics.keys()))
df.head(5)


### O que este código faz
Verifica qualidade básica dos dados: valores ausentes e distribuição da variável-alvo (quando existir). É um check rápido antes de qualquer análise.


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print('missing columns:', len(missing))
if len(missing) > 0:
    display(missing)

if 'attrition' in df.columns:
    print('target distribution:')
    print(df['attrition'].value_counts(normalize=True).round(4))


### O que este código faz
Realiza uma análise exploratória específica do projeto para gerar visão inicial dos padrões principais.


In [ ]:
group_cols = [c for c in ['department', 'seniority', 'work_mode'] if c in df.columns]
for col in group_cols:
    risk = df.groupby(col)['attrition'].mean().sort_values(ascending=False)
    print(f'\nAttrition rate by {col}:')
    print(risk.round(4))


### O que este código faz
Gera visualizações complementares para facilitar interpretação e comunicação dos resultados.


In [ ]:
for col in ['engagement_score', 'overtime_hours_month', 'salary_market_ratio']:
    if col in df.columns:
        plt.figure(figsize=(5.2, 3.2))
        plt.hist(df.loc[df['attrition']==0, col], bins=25, alpha=0.7, label='stay')
        plt.hist(df.loc[df['attrition']==1, col], bins=25, alpha=0.7, label='attrition')
        plt.title(col)
        plt.legend()
        plt.grid(alpha=0.2)
        plt.tight_layout()
        plt.show()


### O que este código faz
Exibe as métricas salvas no pipeline para conectar análise exploratória com desempenho final do modelo.


In [ ]:
pd.DataFrame([metrics]).T.rename(columns={0: 'value'}).head(30)


## Status atual
- Este projeto ja foi executado de ponta a ponta.
- Artefatos (dados, modelos, metricas e relatorios) estao gerados.
- Repositorio publicado e sincronizado no GitHub.
